In [ ]:
import pandas as pd

In [ ]:
cleaned_data = pd.read_csv("../../../data/balanced_dataset_30k.csv")

In [ ]:
cleaned_data.head()

In [ ]:
from sklearn.model_selection import train_test_split

# Split text and labels first to avoid fitting the vectorizer on test data.
X = cleaned_data["text"]
y = cleaned_data["label"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

In [ ]:
from src.training.preprocess import preprocess_text

# Apply preprocessing after split
X_train_preprocessed = X_train.apply(lambda x: preprocess_text(x))
X_test_preprocessed = X_test.apply(lambda x: preprocess_text(x))

print(f"Original X_train[0]:\n{X_train.iloc[0][:200]}\n")
print(f"Preprocessed X_train[0]:\n{X_train_preprocessed.iloc[0][:200]}")

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Fit TF-IDF on preprocessed training text only, then transform test text.
vectorizer = TfidfVectorizer(
    max_features=10000,
    # stop_words='english',  # Already removed during preprocessing
    # min_df=0.001, max_df=0.999
)

X_train = vectorizer.fit_transform(X_train_preprocessed)
X_test = vectorizer.transform(X_test_preprocessed)
print(type(X_train), X_train.shape, X_test.shape)

BOW_train = pd.DataFrame.sparse.from_spmatrix(
    X_train,
    index=y_train.index,
    columns=vectorizer.get_feature_names_out(),
)

In [ ]:
from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression(max_iter=1000, class_weight="balanced")
log_reg.fit(X_train, y_train)
log_reg.score(X_test, y_test)

In [ ]:
import xgboost as xgb

xgb_model = xgb.XGBClassifier(
    n_estimators=300,  # Plus d'arbres
    max_depth=8,  # Plus profond
    learning_rate=0.05,  # Plus lent mais précis
    subsample=0.8,  # Régularisation
    colsample_bytree=0.8,  # Diversité features
    scale_pos_weight=4.2,  # Ratio 80/20 inversé (0.8078/0.1922)
    random_state=42,
    eval_metric="logloss",
    n_jobs=-1,
)

In [ ]:
model = xgb_model.fit(X_train, y_train, verbose=False)

In [ ]:
import sys
from pathlib import Path

# Ensure the project root (folder containing `src`) is importable in this notebook kernel.
project_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").exists()), None)
if project_root is None:
    raise RuntimeError("Could not locate project root containing `src`.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Using project root: {project_root}")

In [ ]:
from src.dashboard.shap import shap_graph

shap_graph(model=log_reg, X_train=X_train, vectorizer=vectorizer, sample_size=None)

In [ ]:
import shap

# For XGBoost + sparse TF-IDF, do not pass X_train as background masker.
explainer = shap.Explainer(model)
shap_values = explainer(X_train)

feature_names = vectorizer.get_feature_names_out().tolist()
shap_values.feature_names = feature_names

# Global feature importance bar plot
shap.plots.bar(shap_values, max_display=20)

In [ ]:
word_importance = dict(zip(feature_names, shap_values.values[0]))
word_importance_sorted = dict(sorted(word_importance.items(), key=lambda item: abs(item[1]), reverse=True))
word_importance_sorted